<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03d_visualizations_survival_rate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [147]:
!pip install contextily

##Importing

In [148]:

import pandas as pd
import geopandas as gpd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import contextily as cx

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [149]:

!ls /content/drive
!ls /content/drive/Shareddrives

MyDrive  Shareddrives


##Reading in file

In [150]:
gdf = pd.read_csv("/content/drive/MyDrive/C255_final_project/cleaned/rbl_rde_cleaned_all_dates.csv")

##Reading in Census Tracts

In [151]:
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]

# eproject to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

In [152]:
import plotly.express as px

##Reading in Census Tracts

In [153]:
tracts_gdf = gpd.read_file(
    "/content/drive/MyDrive/C255_final_project/cb_2020_06_tract_500k"
)
# simplifying
tracts_gdf = tracts_gdf[["NAME", "NAMELSAD", "STATE_NAME","GEOID", "geometry"]]
#troubleshooting
tracts_gdf = tracts_gdf.drop(columns=["index_right", ], errors="ignore")
gdf = gdf.drop(columns=["index_right"], errors="ignore")

##Joining census tracts

In [154]:
gdf = gdf.set_geometry(
    gpd.points_from_xy(gdf.lon, gdf.lat)
)

gdf = gdf.set_crs(epsg=4326)
tracts_gdf = tracts_gdf.to_crs(epsg=4326)

# joining within
business_tracts_gdf = gpd.sjoin(gdf, tracts_gdf, how="left", predicate="within")

##Limiting columns

In [155]:
business_tracts_gdf.columns.tolist()

cols_to_keep = [
    "dba_name",
    "uniqueid",
    "naics_code",
    "year",
    "status",
    "location_start_date",
    "location_end_date",
    "GEOID",
    "geometry",
    "lon", "lat"
]

business_tracts_gdf_clipped = business_tracts_gdf[cols_to_keep]

business_tracts_gdf_clipped["location_start_date"] = pd.to_datetime(
    business_tracts_gdf_clipped["location_start_date"]
)
business_tracts_gdf_clipped["location_end_date"] = pd.to_datetime(
    business_tracts_gdf_clipped["location_end_date"]
)

/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/usr/local/lib/python3.12/dist-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [164]:
survival_mask = (business_tracts_gdf_clipped['location_start_date'].dt.year < 2020) & business_tracts_gdf_clipped['location_end_date'].isna()
survival_tracts = business_tracts_gdf_clipped[survival_mask].sort_values('location_start_date')

In [ ]:
survival_tracts_grouped = (
    survival_tracts(['GEOID', 'year', ''])
    .groupby
)

In [157]:
# I'm counting the number of businesses per tract per year, separated by closing and openings status
# tract_year = (
#     business_tracts_gdf_clipped
#     .groupby(["GEOID", "year", "status"])
#     .size()
#     .reset_index(name="count")
#     .pivot(index=["GEOID", "year"], columns="status", values="count")
#     .fillna(0)
#     .reset_index()
# )
# tract_year

In [158]:
# tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
#     tract_year,
#     on="GEOID",
#     how="left"
# ).fillna(0)

# #clipping to sf only
# tracts_plot.info()
# tracts_plot = gpd.clip(tracts_plot, sf_poly)

In [159]:
epc_tracts = gpd.read_file("/content/drive/MyDrive/C255_final_project/epc_tracts.geojson")

In [160]:
epc_tracts = gpd.clip(epc_tracts, sf_poly)
epc_tracts = epc_tracts.to_crs('EPSG:4326')